<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/Playground/s6e5/1_XGBoost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [33]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
df_test = pd.read_csv('/content/drive/MyDrive/DATASET/playground-series-s6e5/test.csv')
df_train = pd.read_csv('/content/drive/MyDrive/DATASET/playground-series-s6e5/train.csv')

In [35]:
df_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [36]:
df_test.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
0,439140,D119,MEDIUM,British Grand Prix,2023,0,21,1,21.0,4,93.387,0.280,-4.984,0.403846,0.0
1,439141,VER,MEDIUM,Abu Dhabi Grand Prix,2023,0,24,1,24.0,1,90.867,-0.129,-1.990,0.413793,0.0
2,439142,D270,MEDIUM,British Grand Prix,2023,0,24,1,24.0,11,92.871,0.041,-8.842,0.461538,0.0
3,439143,D112,SOFT,São Paulo Grand Prix,2024,0,6,2,4.0,15,94.967,-19.741,8.250,0.077922,1.0
4,439144,AND,HARD,United States Grand Prix,2024,0,52,2,29.0,12,99.112,0.930,-20.848,0.722222,7.0


tanpa cleaning

In [37]:
import xgboost as xgb

In [38]:
X = df_train.drop(['id','PitNextLap'], axis=1)
y = df_train['PitNextLap']

In [39]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=10)

In [40]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=10
)

In [41]:
model.fit(X_train, y_train)
#akan terjadi eror

ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Driver: object, Compound: object, Race: object

Lakukan Clening

In [42]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [43]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 188165 entries, 0 to 188164
Data columns (total 15 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      188165 non-null  int64  
 1   Driver                  188165 non-null  object 
 2   Compound                188165 non-null  object 
 3   Race                    188165 non-null  object 
 4   Year                    188165 non-null  int64  
 5   PitStop                 188165 non-null  int64  
 6   LapNumber               188165 non-null  int64  
 7   Stint                   188165 non-null  int64  
 8   TyreLife                188165 non-null  float64
 9   Position                188165 non-null  int64  
 10  LapTime (s)             188165 non-null  float64
 11  LapTime_Delta           188165 non-null  float64
 12  Cumulative_Degradation  188165 non-null  float64
 13  RaceProgress            188165 non-null  float64
 14  Position_Change     

In [44]:
df_train['Compound'].value_counts()

,count
Compound,
MEDIUM,211141
HARD,170518
SOFT,38744
INTERMEDIATE,17382
WET,1355


In [45]:
df_test['Compound'].value_counts()

,count
Compound,
MEDIUM,90897
HARD,72677
SOFT,16615
INTERMEDIATE,7408
WET,568


In [46]:
mapping={
    'WET':0,
    'MEDIUM':2,
    'SOFT':1,
    'INTERMEDIATE':3,
    'HARD':4
}

In [47]:
df_train['Compound'] = df_train['Compound'].map(mapping)
df_test['Compound'] = df_test['Compound'].map(mapping)


In [48]:
df_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,4,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,4,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,4,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,2,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,4,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [52]:
df_train['Driver'].value_counts()

,count
Driver,
MAS,1682
RAI,1669
BAR,1656
BUT,1655
FIS,1651
...,...
D723,1
D677,1
D731,1


In [53]:
from sklearn.preprocessing import LabelEncoder

In [54]:
df_train['Driver'] = LabelEncoder().fit_transform(df_train['Driver'])
df_test['Driver'] = LabelEncoder().fit_transform(df_test['Driver'])

In [55]:
df_train['Race'].value_counts()

,count
Race,
Dutch Grand Prix,24462
Mexico City Grand Prix,23672
Pre-Season Testing,22492
Hungarian Grand Prix,22481
Monaco Grand Prix,21539
Canadian Grand Prix,21416
Austrian Grand Prix,21223
Spanish Grand Prix,20483
Italian Grand Prix,19854


In [56]:
df_train['Race'] = LabelEncoder().fit_transform(df_train['Race'])
df_test['Race'] = LabelEncoder().fit_transform(df_test['Race'])

In [49]:
# df_train = pd.get_dummies(df_train, columns=['Driver','Race'],drop_first=True,dtype=int)
# df_test = pd.get_dummies(df_test, columns=['Driver','Race'],drop_first=True,dtype=int)

In [57]:
df_train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,134,4,7,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,111,4,9,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,886,4,2,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,864,2,19,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,44,4,3,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [58]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  int64  
 2   Compound                439140 non-null  int64  
 3   Race                    439140 non-null  int64  
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [59]:
df_train.set_index('id',inplace=True)
df_test.set_index('id',inplace=True)

In [60]:
X = df_train.drop(['PitNextLap'], axis=1)
y = df_train['PitNextLap']

In [61]:
X_train,_X_test,y_train,_y_test = train_test_split(
    X,y,test_size=0.2,random_state=10)

In [63]:
model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=10
)

In [65]:
model.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [67]:
y_pred = model.predict(_X_test)


In [68]:
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report

In [69]:
print(accuracy_score(_y_test,y_pred))


0.8948968438311244


In [70]:
print(confusion_matrix(_y_test,y_pred))


[[66079  4344]
 [ 4887 12518]]


In [71]:
y_submit = model.predict(df_test)

In [72]:
submission = pd.DataFrame({
    'id':df_test.index,
    'PitNextLap':y_submit
})

In [75]:
import numpy as np

In [78]:
submission.to_csv('submission.csv',index=False)

Coba Optuna Tuning

In [79]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 7.3 MB/s eta 0:00:00


In [80]:
import optuna

In [81]:
from sklearn.model_selection import cross_val_score

In [82]:
def objective(trial):
  param = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42
  }

  model = xgb.XGBClassifier(**param)
  score = cross_val_score(model,X_train,y_train,cv=5,scoring='accuracy')
  accuracy = score.mean()
  return accuracy

In [83]:
study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=50)

[I 2026-05-05 12:44:48,057] A new study created in memory with name: no-name-104a06a4-4aad-49c0-aa93-7014e749f21c
[I 2026-05-05 12:45:49,670] Trial 0 finished with value: 0.8966787305930805 and parameters: {'n_estimators': 329, 'max_depth': 6, 'learning_rate': 0.11309039567172086, 'subsample': 0.8259556997413092, 'colsample_bytree': 0.6113415720576331}. Best is trial 0 with value: 0.8966787305930805.
[I 2026-05-05 12:47:57,936] Trial 1 finished with value: 0.8966929669670163 and parameters: {'n_estimators': 331, 'max_depth': 10, 'learning_rate': 0.021158655705002673, 'subsample': 0.8702370423098942, 'colsample_bytree': 0.9298859686420008}. Best is trial 1 with value: 0.8966929669670163.
[I 2026-05-05 12:49:29,699] Trial 2 finished with value: 0.8972195625844573 and parameters: {'n_estimators': 479, 'max_depth': 7, 'learning_rate': 0.11317625637665447, 'subsample': 0.7796585732150322, 'colsample_bytree': 0.8808138622940607}. Best is trial 2 with value: 0.8972195625844573.
[I 2026-05-05 

In [85]:
print(study.best_params)

{'n_estimators': 412, 'max_depth': 10, 'learning_rate': 0.055582681409380254, 'subsample': 0.967063161179112, 'colsample_bytree': 0.9804913252113854}


In [86]:
print("Best score:", study.best_value)

Best score: 0.8983780758110717


In [88]:
best_model = xgb.XGBClassifier(**study.best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(_X_test)

In [89]:
y_submit = best_model.predict(df_test)

In [90]:
submission = pd.DataFrame({
    'id':df_test.index,
    'PitNextLap':y_submit
})

In [91]:
submission.to_csv('submission_2.csv',index=False)